In [1]:
import pandas as pd
from pathlib import Path

source_file = Path("FY_2017_Tech_Doc_0.csv")

df = pd.read_csv(
    source_file,
    header=None,
    names=["line"],
    dtype=str,
    engine="python"
)

df["line"] = (
    df["line"]
    .fillna("")
    .str.replace("\x0c", "", regex=False)
    .str.strip()
)

sections = {
    "UNIT_Demo.csv": (
        "Unit demographics and sample weights",
        "Unit countable income"
    ),
    "UNIT_Inc.csv": (
        "Unit countable income",
        "Unit countable and reported assets"
    ),
    "UNIT_Assets.csv": (
        "Unit countable and reported assets",
        "Unit expenses and deductions"
    ),
    "UNIT_ExDed.csv": (
        "Unit expenses and deductions",
        "Unit benefits"
    ),
    "PERS_Char.csv": (
        "Person-level characteristics",
        "Person-level countable income"
    ),
    "PERS_Inc.csv": (
        "Person-level countable income",
        "Detailed error findings"
    ),
}

quick_ref_start = df.index[
    df["line"].str.contains("Quick-Reference Codebook", case=False, regex=False)
][0]

def find_row(text, start=0):
    matches = df.index[
        df["line"].str.contains(text, case=False, regex=False)
    ]
    matches = [i for i in matches if i >= start]
    return matches[0]

for output_name, (start_text, end_text) in sections.items():
    start = find_row(start_text, quick_ref_start)
    end = find_row(end_text, start + 1)

    temp_df = df.loc[start:end-1, "line"].copy()

    temp_df = temp_df[
        temp_df.str.match(r"^[A-Za-z0-9_]+i?\s+[CR]\s+")
    ]

    temp_df = temp_df.str.split().str[0]

    output_path = source_file.parent / output_name

    pd.DataFrame({"Table 1": temp_df}).to_csv(
        output_path,
        index=False
    )

    print(f"Created {output_name}: {len(temp_df)} rows")

Created UNIT_Demo.csv: 30 rows
Created UNIT_Inc.csv: 27 rows
Created UNIT_Assets.csv: 11 rows
Created UNIT_ExDed.csv: 27 rows
Created PERS_Char.csv: 17 rows
Created PERS_Inc.csv: 21 rows
